# Estimate bivariate equity–housing AR(1)

Pipeline aligned with the model (MODEL.md §7):

1. Year-end **real price indices** (default equity = **50/50 S&P500 CAD + TSX60**)
2. **Log-difference** → annual log growth
3. Save **mean log growth** μ
4. **Demean**, then diagonal bivariate AR(1) on ε

Also estimate the older **simple-return** AR(1) for comparison.

Artifact: ../output/macro_returns/bivar_ar1_canada_return_params.json


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))

from pull_macro_returns import annualize_returns, build_panel
from src.returns_estimate import (
    annual_real_indices,
    compare_estimate_dicts,
    estimate_bivar_ar1_from_indices,
    estimate_diag_ar1_simple,
    estimates_to_canada_return_params,
    log_diff_growth,
    save_return_estimates,
)

OUT = ROOT / "output" / "macro_returns"
OUT.mkdir(parents=True, exist_ok=True)
print("ROOT", ROOT)

## 1. Load or refresh macro panel

Uses cached `macro_returns_monthly.csv` when present; otherwise downloads (network).

In [ ]:
monthly_path = OUT / "macro_returns_monthly.csv"
if monthly_path.is_file():
    df = pd.read_csv(monthly_path, index_col=0, parse_dates=True)
    print("Loaded cache", monthly_path, df.shape)
else:
    print("No cache; downloading macro panel (may take ~30s)...")
    df = build_panel()
    df.to_csv(monthly_path)
    print("Wrote", monthly_path)

ann_simple = annualize_returns(df)
ann_simple.to_csv(OUT / "macro_returns_annual.csv")
display(ann_simple.tail())

## 2. Real indices → log growth → mean → demean → AR(1)

Year-end levels of `equity_comb_real` (50/50 S&P500 CAD + TSX60; default) and `house_real`, then:

$$
g^i_t = \Delta \log P^i_t,\qquad
\mu_i = \overline{g^i},\qquad
\varepsilon^i_t = g^i_t - \mu_i.
$$

Diagonal AR(1) on $(\varepsilon^f,\varepsilon^h)$.

In [ ]:
idx = annual_real_indices(df)
idx.to_csv(OUT / "annual_real_price_indices.csv")
display(idx.tail())

g_f = log_diff_growth(idx["P_equity"])
g_h = log_diff_growth(idx["P_house"])
growth = pd.DataFrame(
    {"g_equity": g_f, "g_house": g_h},
    index=idx.index[1:],
)
growth.to_csv(OUT / "annual_log_growth.csv")

mean_log_f = float(growth["g_equity"].mean())
mean_log_h = float(growth["g_house"].mean())
print(f"mean log growth equity  μ_f = {mean_log_f:.6f}  (exp(μ)={np.exp(mean_log_f):.6f})")
print(f"mean log growth housing μ_h = {mean_log_h:.6f}  (exp(μ)={np.exp(mean_log_h):.6f})")

demeaned = growth.assign(
    eps_equity=growth["g_equity"] - mean_log_f,
    eps_house=growth["g_house"] - mean_log_h,
)
display(demeaned.describe().T)

est_log = estimate_bivar_ar1_from_indices(idx["P_equity"], idx["P_house"])
display(pd.Series(est_log, name="log_index_AR1").to_frame())

params = estimates_to_canada_return_params(est_log, n_f=3, n_h=3)
print("Mapped CanadaReturnParams (Rf_bar = mean exp(Δlog P)):")
print(params)

## 3. Compare to simple-return AR(1)

Same overlapping annual sample idea, but OLS on simple real returns
`r = ∏(1+r_m)−1` from the monthly panel (previous notebook path).

In [ ]:
est_simple = estimate_diag_ar1_simple(
    ann_simple["r_equity"].to_numpy(),
    ann_simple["r_house"].to_numpy(),
)
cmp = compare_estimate_dicts(est_log, est_simple)
display(cmp)

out_path = save_return_estimates(
    est_log,
    OUT / "bivar_ar1_canada_return_params.json",
    n_f=3,
    n_h=3,
    comparison=est_simple,
    meta={
        "equity": "equity_comb_real year-end (50/50 S&P500 CAD + TSX60)",
        "housing": "house_real year-end (BIS/CPI)",
        "frequency": "annual",
        "returns": "Δlog real price index; AR(1) on demeaned log growth",
        "Rf_bar_mapping": "mean(exp(Δlog P))",
    },
)
print("Saved", out_path)
pd.Series(est_log).to_csv(OUT / "bivar_ar1_equity_housing.csv", header=["value"])

## Parameter correspondence

| Saved object | → `CanadaReturnParams` |
|--------------|------------------------|
| `mean_log_f`, `mean_log_h` | kept in JSON (mean log growth); not bars directly |
| `mean_Rf_gross` = mean(exp(Δlog P^f)) | `Rf_bar` |
| `mean_Rh_gross` = mean(exp(Δlog P^h)) | `Rh_bar` |
| `rho_f`, `rho_h` on demeaned log growth | `rho_f`, `rho_h` |
| `sigma_f`, `sigma_h` innov. SD | `sigma_f`, `sigma_h` |
| `corr_innov` | `corr` |

Demo notebook loads via `load_canada_return_params()`; mean logs via `load_mean_log_growth()`.